In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

In [3]:
num_customers = 1000

customers = pd.DataFrame({
    "customer_id": range(1, num_customers + 1)
})

In [4]:
customers["signup_date"] = np.random.choice(
    pd.date_range("2023-01-01", "2024-01-01"),
    num_customers
)

In [5]:
customers.head()

,customer_id,signup_date
0,1,2023-04-13
1,2,2023-12-15
2,3,2023-09-28
3,4,2023-04-17
4,5,2023-03-13


In [6]:
orders_list = []
order_id = 1

In [7]:
categories = ["Dresses", "Shoes", "Accessories", "Tops"]

In [8]:
print(categories)

['Dresses', 'Shoes', 'Accessories', 'Tops']


In [9]:
orders_list = []
order_id = 1

for customer in customers["customer_id"]:
    
    num_orders = np.random.randint(1, 6)
    
    for _ in range(num_orders):
        
        order_date = np.random.choice(
            pd.date_range("2024-01-01", "2024-12-31")
        )

        channel = np.random.choice(
            ["Online", "Store"],
            p=[0.7, 0.3]
        )

        category = np.random.choice(categories)

        price = round(np.random.uniform(10, 200), 2)

        discount = np.random.choice([0, 10, 20], p=[0.6, 0.3, 0.1])

        orders_list.append([
            order_id,
            customer,
            order_date,
            channel,
            category,
            price,
            discount
        ])

        order_id += 1

In [10]:
orders = pd.DataFrame(orders_list, columns=[
    "order_id",
    "customer_id",
    "order_date",
    "channel",
    "category",
    "price",
    "discount"
])

In [11]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2994 entries, 0 to 2993
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     2994 non-null   int64         
 1   customer_id  2994 non-null   int64         
 2   order_date   2994 non-null   datetime64[ns]
 3   channel      2994 non-null   object        
 4   category     2994 non-null   object        
 5   price        2994 non-null   float64       
 6   discount     2994 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 163.9+ KB


In [12]:
len(orders_list)

2994

In [13]:
print(orders.head())

   order_id  customer_id order_date channel category   price  discount
0         1            1 2024-11-28  Online  Dresses   80.15         0
1         2            1 2024-09-17  Online     Tops   48.33        10
2         3            1 2024-03-24  Online  Dresses   10.52         0
3         4            1 2024-02-07  Online    Shoes   30.27        10
4         5            2 2024-08-23  Online  Dresses  148.86        20


In [14]:
def get_return_prob(row):
    if row["channel"]== "Online":
        return 0.35
    else:
        return 0.15

In [15]:
orders["is_returned"] = orders.apply(
    lambda row: np.random.choice(
        [1, 0],
        p = [get_return_prob(row), 1 - get_return_prob(row)]
    ),
    axis = 1
)

In [16]:
orders["days_to_return"] = np.where(
    orders["is_returned"] == 1,
    np.random.randint(1, 28, len(orders)),
    None
)

In [17]:
from datetime import timedelta

orders["return_date"] = orders.apply(
    lambda row: row["order_date"] + timedelta(days= int(row["days_to_return"]))
    if row["is_returned"] == 1 else None,
    axis=1
)

In [18]:
customer_returns = orders.groupby("customer_id")["is_returned"].mean().reset_index()

customer_returns.rename(columns={"is_returned": "return_rate"}, inplace=True)

In [19]:
orders.head()

,order_id,customer_id,order_date,channel,category,price,discount,is_returned,days_to_return,return_date
0,1,1,2024-11-28,Online,Dresses,80.15,0,1,20,2024-12-18
1,2,1,2024-09-17,Online,Tops,48.33,10,1,7,2024-09-24
2,3,1,2024-03-24,Online,Dresses,10.52,0,0,None,NaT
3,4,1,2024-02-07,Online,Shoes,30.27,10,1,25,2024-03-03
4,5,2,2024-08-23,Online,Dresses,148.86,20,0,None,NaT


In [20]:
orders["is_returned"].mean()

np.float64(0.28122912491649965)

In [21]:
orders.groupby("channel")["is_returned"].mean()

channel
Online    0.341348
Store     0.135166
Name: is_returned, dtype: float64

In [22]:
def segment_customer(rate):
    if rate < 0.2:
        return "Low Return"
    elif rate < 0.6:
        return "Medium Return"
    else:
        return "High Return"

In [23]:
customer_returns["segment"] = customer_returns["return_rate"].apply(segment_customer)

customer_returns["segment"].value_counts()

segment
Medium Return    421
Low Return       413
High Return      166
Name: count, dtype: int64

In [24]:
customer_returns.head()

,customer_id,return_rate,segment
0,1,0.75,High Return
1,2,0.00,Low Return
2,3,0.00,Low Return
3,4,1.00,High Return
4,5,0.50,Medium Return


In [25]:
customer_returns["segment"].value_counts()

segment
Medium Return    421
Low Return       413
High Return      166
Name: count, dtype: int64

In [26]:
customer_returns["eligible_for_reward"] = customer_returns["segment"] == "Low Return"

In [27]:
customer_returns["eligible_for_reward"].value_counts()

eligible_for_reward
False    587
True     413
Name: count, dtype: int64

In [28]:
customer_returns["eligible_for_reward"].mean()

np.float64(0.413)

In [29]:
orders.to_csv("retail_returns.csv", index=False)

In [30]:
import os
os.getcwd()

'C:\\Users\\Nares'

In [31]:
os.listdir()

['.atom',
 '.idlerc',
 '.ipynb_checkpoints',
 '.ipython',
 '.jupyter',
 '.matplotlib',
 'age_wise_data_transformed.csv',
 'age_wise_data_transformed.xlsx',
 'AppData',
 'Application Data',
 'beginner_project_1.ipynb',
 'comprehension.py',
 'Contacts',
 'Cookies',
 'Documents',
 'Downloads',
 'Favorites',
 'interview practise.py',
 'Links',
 'Local Settings',
 'Mental_health_project.ipynb',
 'Microsoft',
 'Music',
 'My Documents',
 'my-portfolio',
 'my_portfolio',
 'NetHood',
 'NTUSER.DAT',
 'ntuser.dat.LOG1',
 'ntuser.dat.LOG2',
 'NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TM.blf',
 'NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TMContainer00000000000000000001.regtrans-ms',
 'NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TMContainer00000000000000000002.regtrans-ms',
 'ntuser.ini',
 'OneDrive',
 'PrintHood',
 'py.py',
 'Recent',
 'Retail project.ipynb',
 'retail_returns.csv',
 'Saved Games',
 'Searches',
 'SendTo',
 'Start Menu',
 'Templates',
 'titanic_ml_project.ipynb',
 '